In [ ]:
!conda install ray[cgraph]

In [ ]:
import os
import time

import matplotlib.pyplot as plt
import pandas as pd
import ray
import seaborn as sns
import torch
from tqdm import tqdm

# Connect to the existing Ray cluster
ray.init(address="auto", ignore_reinit_error=True)

# 1 GB float32 tensor
TENSOR_SIZE_BYTES = 1 * 1024**3
TENSOR_SHAPE = (250_000_000,)

print(f"Ray Cluster Resources: {ray.available_resources()}")
print(f"Testing with Payload Size: {TENSOR_SIZE_BYTES / 1e9:.2f} GB")

# Set up plotting style for academic/HPC reporting
sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)

In [ ]:
@ray.remote(num_gpus=1)
class Sender:
    def __init__(self, shape):
        self.shape = shape
        self.tensor = torch.ones(self.shape, dtype=torch.float32, device="cuda")

    def get_data(self, trigger_input):  # <-- FIX: Added dummy parameter
        return self.tensor


@ray.remote(num_gpus=1)
class Receiver:
    def __init__(self):
        self.sink = None

    def consume_data(self, tensor):
        self.sink = tensor
        return True


# Instantiate the actors
sender = Sender.remote(TENSOR_SHAPE)
receiver = Receiver.remote()

# Warm up GPUs (pass a 0 as the dummy trigger)
ray.get(receiver.consume_data.remote(sender.get_data.remote(0)))
print("Actors initialized and warmed up.")

In [ ]:
def benchmark_object_store(iterations=20):
    records = []
    print(f"Running Baseline (Object Store) for {iterations} iterations...")

    for i in range(iterations):
        start_time = time.perf_counter()

        # Standard Ray object store transfer — inter-node when actors land on separate nodes
        data_ref = sender.get_data.remote(0)
        ready_ref = receiver.consume_data.remote(data_ref)
        ray.get(ready_ref)

        end_time = time.perf_counter()
        iteration_time = end_time - start_time

        records.append(
            {
                "Method": "Ray Object Store (inter-node)",
                "Iteration": i + 1,
                "Latency (ms)": iteration_time * 1000,
                "Throughput (GB/s)": (TENSOR_SIZE_BYTES / 1e9) / iteration_time,
            }
        )

    return pd.DataFrame(records)


df_baseline = benchmark_object_store(iterations=20)

In [ ]:
# No NodeAffinitySchedulingStrategy — actors land on separate nodes automatically when the
# cluster has 1 GPU per node, measuring true inter-node 400G RDMA transfer performance.
# (The old same-node pin was measuring intra-node NVLink/PCIe, not the RDMA fabric.)
print(f"Sender:   {sender}")
print(f"Receiver: {receiver}")

In [ ]:
from ray.dag.input_node import InputNode

# Define and compile the graph with NCCL tensor transport on the GPU→GPU edge.
# .with_tensor_transport("nccl") tells the compiled graph to use NCCL/RDMA for this
# data edge instead of the default Ray object store path (TCP).
with InputNode() as inp:
    data = sender.get_data.bind(inp)
    result = receiver.consume_data.bind(data.with_tensor_transport("nccl"))

compiled_dag = result.experimental_compile()
print(compiled_dag.visualize(format="ascii", view=True))

# Warmup pass (not measured)
ray.get(compiled_dag.execute(0))


def benchmark_compiled_graph(iterations=20):
    records = []
    print(f"Running Compiled Graph (NCCL/RDMA) for {iterations} iterations...")

    for i in tqdm(range(iterations)):
        start_time = time.perf_counter()

        ray.get(compiled_dag.execute(i))

        end_time = time.perf_counter()
        iteration_time = end_time - start_time

        records.append(
            {
                "Method": "Compiled Graph (NCCL/RDMA)",
                "Iteration": i + 1,
                "Latency (ms)": iteration_time * 1000,
                "Throughput (GB/s)": (TENSOR_SIZE_BYTES / 1e9) / iteration_time,
            }
        )

    return pd.DataFrame(records)


df_nccl = benchmark_compiled_graph(iterations=20)

df_results = pd.concat([df_baseline, df_nccl], ignore_index=True)
print("Benchmarking complete.")

In [ ]:
def plot_benchmark_results(df):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(
        f"GPU-to-GPU Transfer Performance (Payload: {TENSOR_SIZE_BYTES / 1e9:.1f} GB Tensor)",
        fontweight="bold",
    )

    # Plot 1: Throughput Distribution (Violin Plot)
    # This highlights stability and variance across iterations
    sns.violinplot(
        data=df,
        x="Method",
        y="Throughput (GB/s)",
        ax=axes[0],
        palette=["#e74c3c", "#2ecc71"],
        inner="quartile",
    )
    axes[0].set_title("Throughput Distribution per Iteration")
    axes[0].set_ylabel("Throughput (GB/s)")
    axes[0].set_xlabel("")

    # Add a horizontal dashed line representing the theoretical PCIe 4.0 limit (~31.5 GB/s)
    axes[0].axhline(y=31.5, color="gray", linestyle="--", label="PCIe 4.0 x16 Limit (~31.5 GB/s)")
    axes[0].legend()

    # Plot 2: Average Latency (Bar Plot)
    # Shows the raw time cost of a single transfer
    sns.barplot(
        data=df,
        x="Method",
        y="Latency (ms)",
        ax=axes[1],
        palette=["#e74c3c", "#2ecc71"],
        capsize=0.1,
        err_kws={"linewidth": 2},
    )
    axes[1].set_title("Average Transfer Latency (Lower is Better)")
    axes[1].set_ylabel("Latency (ms)")
    axes[1].set_xlabel("")

    plt.tight_layout()
    plt.show()

    # Output the raw summary statistics for the report
    print("\n--- Summary Statistics ---")
    summary = df.groupby("Method")[["Throughput (GB/s)", "Latency (ms)"]].agg(
        ["mean", "std", "min", "max"]
    )
    print(summary.to_string())


plot_benchmark_results(df_results)